# FoxTrack 02: Baseline early-detection model

This notebook trains a baseline detector on benign traffic only, then measures how many packets pass between the start of each attack and the first alert.

**Before you run it (right panel of the Kaggle editor):**

1. In notebook 01, select **Save Version > Save & Run All (Commit)** and wait until it completes. This keeps the Parquet output.
2. Here, select **Input > Add Input > Notebook Output** and add notebook 01.
3. **Settings > Internet:** set to on.
4. **Private repository only:** attach the `GITHUB_TOKEN` secret to this notebook.

**Method.** Each capture has a long benign period, then one attack start. For each capture:

| Range of the benign period | Use |
|---|---|
| First 70% | Fit a scaler and a PCA (Spark MLlib) |
| Next 15% | Set the alert threshold for a target false positive rate |
| Last 15%, and all packets after it | Test: includes the attack start |

The score of a packet is its PCA reconstruction error. Alert rules: `threshold` (one score more than the threshold), `3 of 5` and `10 of 20` (k of the last n scores), and `mean of 100` (the moving mean of the last 100 scores is more than its threshold).

In [ ]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/baseline"
K_COMPONENTS = 20                      # PCA dimensions
TARGET_FPRS = (1e-2, 1e-3, 1e-4)       # target false positive rates for the threshold
ALERT_RULES = ((1, 1), (3, 5), (10, 20))  # (k, n): k of the last n scores
SMOOTH_WINDOW = 100                    # packets in the moving mean of the score

In [ ]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import anomaly, etl, split
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

## 1. Train and evaluate each capture

In [ ]:
%%time
import os
import mlflow
import pandas as pd

spark = etl.build_spark(driver_memory="20g")
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("baseline_pca_onset")
os.makedirs(f"{OUT_DIR}/scores", exist_ok=True)

all_results = []
for row in split.load_summary(PARQUET_ROOT):
    s = split.onset_split(row)
    capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
    results, test = anomaly.run_capture(capture, s, K_COMPONENTS, TARGET_FPRS, ALERT_RULES, (1, SMOOTH_WINDOW))
    all_results.append(results)
    test.to_parquet(f"{OUT_DIR}/scores/{s.attack}.parquet", index=False)

    with mlflow.start_run(run_name=s.attack):
        mlflow.log_params({"attack": s.attack, "model": "scaler+pca", "k_components": K_COMPONENTS,
                           "train_end": s.train_end, "calib_end": s.calib_end, "onset": s.onset})
        for r in results.itertuples():
            tag = f"fpr{r.target_fpr:g}_{r.alert_rule.replace(' ', '_')}"
            mlflow.log_metric(f"false_alarms_per_100k_{tag}", r.false_alarms_per_100k)
            mlflow.log_metric(f"packet_recall_{tag}", r.packet_recall)
            if r.detected:
                mlflow.log_metric(f"delay_packets_{tag}", r.delay_packets)
    print(f"{s.attack}: done")

results = pd.concat(all_results, ignore_index=True)
results.to_csv(f"{OUT_DIR}/results.csv", index=False)

## 2. Results

**Detection delay in packets** (empty = the attack was not detected). Rows are attacks. Columns are the target false positive rate and the alert rule.

In [ ]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="delay_packets")

**Credible delay in packets.** This table shows a delay only if two checks pass:

- The alert rate on malicious packets is at least 5 times the alert rate on benign packets before the attack.
- A detector with no skill and the same false alarm rate would give an alert that quickly in at most 5% of cases.

An empty cell shows that false alarms can explain the alert. Use this table for the report, not the table of all delays.

In [ ]:
results.assign(credible=results["delay_packets"].where(results["delay_credible"])).pivot(index="attack", columns=["target_fpr", "alert_rule"], values="credible")

**False alarms before the attack start**, for each 100,000 benign packets. A target rate of 0.001 is equal to 100 for each 100,000. A much larger number shows that the benign traffic changed between the training period and the test period.

In [ ]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="false_alarms_per_100k").round(1)

**Packet recall:** the fraction of malicious packets that have an alert.

In [ ]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="packet_recall").round(3)

## 3. Keep the output

Select **Save Version > Save & Run All (Commit)**. The folder `baseline` then has `results.csv` and the test scores for each capture. The file `mlflow.db` has the MLflow records.

To send the results, run this cell and copy the output:

In [ ]:
print(results.to_csv(index=False))